# Barrier Function Class Definition

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import cvxpy as cp
import copy
from typing import Protocol

In [ ]:
class BarrierGeometry(Protocol):
	"""Forces typehinting and barrier geometry type for DECBF object. 
	NOTE: Barrier functions are ONLY well-defined for smooth, continuous functions
	but may be nonlinear.
	PYTHON SPECIFIC"""

	def get_h(self, p_agent:jnp.ndarray) -> jnp.ndarray:
		""" returns geometric expression for barrier eval"""
		pass
	
	def get_center(self) -> jnp.ndarray:
		""" returns center position of the geometry"""
		pass

	def set_center(self) -> None:
		""" sets the center position of the geometry"""
		pass

	def copy(self) -> 'BarrierGeometry':
		""" returns a memory safe copy of the geometry"""
		pass

In [ ]:
class EllipticalGeometry(BarrierGeometry):
	"""
	defines a circular geometry for barrier function creation.
	
	Fields:
		Private _semi_major_axis 	-float, defines the major radius of the geometric object
		Private _semi_minor_axis 	-float, defines the minor radius of the ellipses #RESERVED FOR FUTURE USE
		Private _angle				-float, radians, defines the offset angle of the major axis from x axis #RESERVED FOR FUTURE USE
		Private _center 			-jnp.ndarray, defines the center of the geometric object
		
	Methods:
		Public get_semi_major_axis()	-get major radius value
		Public set_semi_major_axis()	-set major radius value
		Public get_semi_minor_axis()	-get minor radius value #RESERVED FOR FUTURE USE
		Public set_semi_minor_axis()	-set minor radius value #RESERVED FOR FUTURE USE
		Public get_angle()				-get major axis angle from universal x-axis #RESERVED FOR FUTURE USE
		Public set_angle()				-set major axis angle from universal x-axis #RESERVED FOR FUTURE USE
		Public set_relative_angle()		-set major axis angle from relative heading #RESERVED FOR FUTURE USE
		Public get_center()				-get center value
		Public set_center()				-set radius value
		Public get_h()					-defines a geometric definition to autodifferentiate with jax
	"""
	# For reference for C++ conversion
	#_semi_major_axis:float
	#_center:jnp.ndarray
	
	def __init__(self, center: jnp.ndarray = None, semi_major_axis: float = 1.0):
		if (center is not None):
			self._center = jnp.array(center, copy=True) 
		else:
			self._center = jnp.zeros((2,1))

		self._semi_major_axis = semi_major_axis

	def get_semi_major_axis(self) -> float: return self._semi_major_axis
	def set_semi_major_axis(self, semi_major_axis: float): self._semi_major_axis = semi_major_axis
	def get_center(self) -> jnp.ndarray: return jnp.array(self._center, copy=True)
	#RESHAPE SHOULD BE ADJUSTED TO DYNAMIC MODEL (POSITIONAL DIMENSION)
	def set_center(self, center: jnp.ndarray): self._center = jnp.array(center, copy=True).reshape(2,1)

	def copy(self) -> 'EllipticalGeometry':
		"""Returns a memory-safe copy of this object/class"""
		return copy.deepcopy(self)

	def get_h(self, p_agent: jnp.ndarray) -> jnp.ndarray:
		"""Defines the geometry of the barrier for differentiation
		Params:
			p_agent 	- jnp.ndarray, center of the autonomous object
		Returns:
		 	h 		- jnp.ndarray, mathematical definition of the object
		"""
		# ||p_agent - self._center||^2 - self._semi_major_axis^2 = h
		# h is the minimum radius between centers for objects in this definition
		
		# CHANGE FOR ELLIPSE EQUATION IN THE FUTURE, P - shape matrix should be included 
		d_p = p_agent.flatten() - self._center.flatten()
		h = jnp.dot(d_p,d_p) - self._semi_major_axis**2

		return h


## Discrete Exponential CBF Definition (dynamic and static case, circular barriers)
Defining CBF as our control barrier functions around (linearized) 

$$ h(p_{t+1}, t+1) $$

$$ Gp_t \ge (1 - \gamma) * h(p_t) $$

$$ \gamma \in (0,1]$$

Taylor Expansion:

$$h(p_{t+1}, t+1) \approx h(p_t,t) + \frac{\partial h}{\partial p}(p_{t+1}-p_t) + \frac{\partial h}{\partial t} dt $$

By Substitution:

$$ \frac{\partial h}{\partial p} p_{t+1} + [\gamma h(p_t,t) - \frac{\partial h}{\partial p}p_t + \frac{\partial h}{\partial t}dt] \ge 0 $$

Using the gradient defined above:

$$ G = 2[p_t - p_{obs,t}]^T $$

Applyng chain rule to temporal element:

$$ \frac{\partial h}{\partial t} = \frac{\partial h}{\partial p_{obs}} \frac{\partial p_{obs}}{\partial t} = -Gv_{obs}$$

$$ \alpha * dt = \gamma $$

Therefore:

$$ \boxed{ Gp_{t+1} + [\gamma h(p_t, t) - Gp_t - Gv_{obs}dt] \ge 0 } $$


In [ ]:
class DECBF:
	"""Generates a barrier function for controller optimization. 
	This class will accept a position, radius, shape, and alpha value and generate a linearized representation for CVXPY.
	Fields
		Private _alpha  		- float, gradient/softening for barrier function
		Private _geometry		- BarrierGeometry, object defining the barrier's geometry
		Private _heartbeat 		- float, controls simulation clock (ideally a system constant for synchronization)
		Private _velocity		- jnp.ndarray, velocity vector (x,y)
		Private _dt_controller	- float, time between controller updates
		Private _dt_physics		- float, time between simulated continuous physics update
		Private _name			- string, name of cbf for manipulation 

	Methods
		Public linearize() 		- Linearizes the barrier function
		Public get_alpha() 		- returns the softening value
		Public set_alpha() 		- sets the softening value
		Public get_heartbeat() 	- gets the control simulation clock timing
		Public set_heartbeat() 	- sets the control simulation clock timing
		Public get_shape()		- gets barrier shape # RESERVED FOR FUTURE USE (assumed circular for the moment)
		public set_shape()		- sets barrier shape # RESERVED FOR FUTURE USE (assumed circular for the moment)

	Future Efforts:
		Input Validation
		Generalizing Geometry
		Separate and Modularize Obstacle Dynamic Models
		Add Stochastic Elements
"""
	# Field definitions (example)
	#_alpha: float
	#_radius: float
	#_center: jnp.ndarray
	#_heartbeat: float

	def __init__(self, name:str = None, geometry:BarrierGeometry = None, 
			  heartbeat: float = 0.0, physics_clock: float = 0.0, 
			  alpha: float = 0.0, velocity:jnp.ndarray = None):
		
		# Setting fields
		self._name = name
		self._alpha = alpha
		self._physics_clock = physics_clock
		self._heartbeat = heartbeat

		# Setting default values with checking
		if (geometry is not None):
			self._geometry = geometry.copy()
		else:
			self._geometry = None 

		if (velocity is not None):
			self._velocity = velocity
		else:
			self._velocity = None

		if (name is not None):
			self._name = name
		else:
			self._name = None
	
		# Setting clocks
		if (self._physics_clock > 0.0):
				self._dt_physics = 1/self._physics_clock
		else:
			self._dt_physics = 0.0

		if (self._heartbeat > 0.0):
			self._dt_controller = 1/self._heartbeat
		else:
			self._dt_controller = 0.0

	def get_name(self) -> str: return f"{self._name}"
	def set_name(self, name:str) -> None: self._name = f"{name}"

	def get_geometry(self) -> BarrierGeometry: 
		if (self._geometry is None):
			raise ValueError("Geometry uninitialized.")
		return self._geometry.copy()
	
	def set_geometry(self, geometry:'BarrierGeometry') -> None: self._geometry = geometry.copy()

	def get_heartbeat(self) -> jnp.ndarray: return float(self._heartbeat) # generates memory isolated copy
	def set_heartbeat(self, heartbeat:float) -> None: 
		self._heartbeat = heartbeat
		self._dt_controller = 1 / self._heartbeat

	def get_alpha(self) -> jnp.ndarray: return float(self._alpha) # generates memory isolated copy
	def set_alpha(self, alpha:float) ->	None: 
		
		if (alpha > 0 and alpha < 1):
			self._alpha = alpha
		else:
			raise ValueError(f"alpha is {alpha}. alpha must be greater than 0 and less than 1")

	def get_velocity(self) -> jnp.ndarray: return jnp.array(self._velocity, copy=True)
	#RESHAPE SHOULD BE ADJUSTED TO DYNAMIC MODEL (POSITIONAL DIMENSION)
	def set_velocity(self, velocity:jnp.ndarray) -> None: self._velocity = jnp.array(velocity, copy=True).reshape(2,1) 

	def get_physics_clock(self) -> float: return float(self._physics_clock) # generates memory isolated copy
	def set_physics_clock(self, physics_clock:float) -> None: 
		self._physics_clock = physics_clock
		self._dt_physics = 1/ self._physics_clock


	def update_state(self) -> None:
		"""
		Evolves the barrier function's state forward in time for dynamic trajectories
		Params:
			None #IN THE FUTURE INCLUDE A WORLD_CLOCK OR PHYSICS TO CALCULATE ROBUSTLY WITH CLOCKSPEED JITTER
		Returns:
			None
		"""

		if (self._geometry is None):
			raise MemoryError("No geometry set. Set geometry before updating.")
		
		# NOTE: THIS NEEDS TO MANAGE A DYNAMICS OBJECT WHICH DESCRIBES THE EXACT MODEL
		# When that update occurs, physics clock updates will no longer be linear
		# this is constant velocity for simplicity, but more complex dynamics require
		# iterative update for accurate traversal between clock cycles
		temp_center = self._geometry.get_center() + self._velocity*self._dt_physics
		self._geometry.set_center(temp_center)

	def linearize(self, p_agent:jnp.ndarray, p_next:cp.Variable) -> cp.Expression:
		"""
		Returns a linearized version of the barrier function for direct constraint evaluation.

		Params:
			p_agent - jnp.ndarray, cartesian x,y position of autonomous object
			p_next  - cp.Variable, cartesian x,y position for the next step for the automous robot (decision variable)
		Returns:
			result_expression - cp.Expression, Expression to be evaluated with CVXPY
		"""
		
		# Converting to numpy for CVXPY with memory isolation
		G = jax.grad(self._geometry.get_h)(p_agent)

		G_np = np.array(G).flatten()
		p_agent_np = np.array(p_agent).flatten()
		vel_np = np.array(self._velocity).flatten()

		h_np = np.array(self._geometry.get_h(p_agent))

		# Numpy flattening before varible introduction
		offset = (float(self._alpha) * float(self._dt_controller) * h_np 
					- G_np @ p_agent_np 
					- G_np@vel_np*float(self._dt_controller))

		# Internal expression for data isolation and logging #RESERVED FOR FUTURE USE
		internal_expression = G_np @ p_next + offset

		# Converting to numpy for CVXPY
		result_expression = G_np @ p_next + offset
		return result_expression

## CBF Manager
This class manages the cbfs that are called. It will provide a simple interface to pass parameters back and forth, and update states of the functions themselves over time. MPC and agent functions will interface directly with the manage to retrieve and update trajectory information.

In [ ]:
class CBFManager:
	"""
	Generates, manages, and in the future eliminates multiple control barrier functions dynamically across time
	Fields
		Private _cbf_buffer					-list, list of cbfs to 
		Private	_MAX_CBFS					-int, max buffer length for tracked cbfs
		Private _cbf_count					-int, number of cbfs currently tracked
		
	Methods
		Public get_cbf_linearizations()		- returns cbf linearization for cvxpy implementation for trajectories
		Public add_cbf()					- creates a new cbf for simulation with initial position, velocity #RESERVED FOR FUTURE USE - ADD DYNAMICS, SHAPE
		Public get_cbf_list()				- returns a list of the cbfs currently in list
		Private _physics_clock_update()		- updates physics clock on CBF objects
		Private _manage_cbf_states()		- updates the states of the cbfs 
	"""
	
	def __init__(self):
		# Defining maximum number of CBFs
		self._MAX_CBFS = 100

		self._cbf_buffer = [DECBF() for _ in range(self._MAX_CBFS)]
		self._cbf_count = 0

	def get_cbf_linearizations(self, p_agent:jnp.ndarray, p_next: cp.Variable) -> list[tuple]:
		"""Returns python list of cbfs linearizations to use as constraints in cvxpy
		Params:
			p_agent			- jnp.ndarray, position array of the agent
			p_next			- cp.Variable, decision variables of the trajectory optimizer for cvxpy
		Returns:
			cbf_lin_map 	- map, python map of linearization (name, values)
		"""
		# VECTORIZE IN FUTURE UPDATE
		cbf_lin_map = [None] * self._cbf_count
		for i in range(self._cbf_count):
			cbf_lin_map[i] = (self._cbf_buffer[i].get_name(), self._cbf_buffer[i].linearize(p_agent, p_next))
		
		return cbf_lin_map

	def get_cbf_names(self) -> list[str]:
		"""Returns a python list copying the list of cbfs
		Params:
			None
		Return:
			cbf_names - list, python list of cbf names
		"""
		
		# VECTORIZE IN FUTURE UPDATE
		cbf_names = [""] * self._cbf_count
		for i in range(self._cbf_count):
			cbf_names[i] = self._cbf_buffer[i].get_name()

		return cbf_names
		
	# RESERVED FOR FUTURE UPDATE. FOR NOW, EVERYTHING RUNS ON THE SAME CLOCK
	#def _physics_clock_update():
	#	"""Updates world clock within cbfs"""
	#	...

	#def _manage_cbf_states():
	#	""" Update cbf states according to controller clock"""
	#	...
	

	def add_cbf(self, name:str, geometry: BarrierGeometry, heartbeat:float, 
			 physics_clock:float, alpha:float, velocity:jnp.ndarray) -> None:
		"""Adds a cbf to the manager buffer
		Params:
			geometry 		- BarrierGeometry, geometric definition of barrier
			heatbeat 		- float, controller clockspeed
			physics_clock 	- float, world clockspeed
			alpha			- float, barrier softening function
			velocity		- jnp.ndarray, velocity array
		Returns:
			None
		"""

		#buffer check
		if self._cbf_count >= self._MAX_CBFS:
			raise MemoryError("Buffer overflow: Max CBFS exceeded")
		
		#C++: cbf_buffer[cbf_count] = cbf(...)
		self._cbf_buffer[self._cbf_count].set_name(name)
		self._cbf_buffer[self._cbf_count].set_geometry(geometry)
		self._cbf_buffer[self._cbf_count].set_heartbeat(heartbeat)
		self._cbf_buffer[self._cbf_count].set_physics_clock(physics_clock)
		self._cbf_buffer[self._cbf_count].set_alpha(alpha)
		self._cbf_buffer[self._cbf_count].set_velocity(velocity)

		#Update counter
		self._cbf_count += 1

	def remove_cbf(self, name:str) -> None:
		"""removes a cbf to the manager buffer
		Params:
			cbf 		- DECBF, cbf to remove
		Returns:
			None
		"""

		#buffer check
		if self._cbf_count < 1:
			raise MemoryError("No CBFs in buffer. Please add cbf before removing")
		
		target_i = -1

		for i in range(self._cbf_count):
			if (self._cbf_buffer[i].get_name() == name):
				target_i = i
				break
		
		if (target_i == -1):
			raise ValueError(f"No CBF with that '{name}' in buffer. Please add CBF with that name to remove")

		last_valid_index = self._cbf_count - 1

		if (last_valid_index != target_i):
			# Swapping last element from data field
			last_element = self._cbf_buffer[last_valid_index]

			self._cbf_buffer[target_i].set_name(last_element.get_name())
			self._cbf_buffer[target_i].set_geometry(last_element.get_geometry())
			self._cbf_buffer[target_i].set_heartbeat(last_element.get_heartbeat())
			self._cbf_buffer[target_i].set_physics_clock(last_element.get_physics_clock())
			self._cbf_buffer[target_i].set_alpha(last_element.get_alpha())
			self._cbf_buffer[target_i].set_velocity(last_element.get_velocity())

		self._cbf_buffer[last_valid_index] = DECBF()
		self._cbf_count -= 1


		



## Verification Tests
### Code Architecture Test

In [ ]:
def run_cbf_architecture_test():
    print("=== Initializing CBF Architecture Test ===\n")
    manager = CBFManager()

    # 1. Define Agent State (Origin) and CVXPY Optimizer Variable
    p_agent = jnp.array([[0.0], [0.0]])
    p_next = cp.Variable(2) # 1D Vector for CVXPY stability

    print("=== Phase 1: Injecting Barriers ===")
    
    # A. Static Barrier (obstacle/Boundary)
    geom_static = EllipticalGeometry(center=jnp.array([[5.0], [5.0]]), semi_major_axis=2.0)
    manager.add_cbf(
        name="Static_obstacle_1",
        geometry=geom_static,
        heartbeat=10.0,       # 10 Hz Controller
        physics_clock=10.0,   # 10 Hz World Clock
        alpha=0.5,
        velocity=jnp.array([[0.0], [0.0]]) # Not moving
    )
    print(f"Added: Static_obstacle_1 at {geom_static.get_center().flatten()}")

    # B. Dynamic Barrier (Moving Obstacle)
    geom_dynamic = EllipticalGeometry(center=jnp.array([[10.0], [0.0]]), semi_major_axis=1.5)
    manager.add_cbf(
        name="Dynamic_Obstacle_1",
        geometry=geom_dynamic,
        heartbeat=10.0,
        physics_clock=10.0,
        alpha=0.8,
        velocity=jnp.array([[-2.0], [1.0]]) # Moving Left and Up
    )
    print(f"Added: Dynamic_Obstacle_1 at {geom_dynamic.get_center().flatten()}")
    print(f"Active CBFs Tracked: {manager._cbf_count}\n")


    print("=== Phase 2: Simulation Loop & CVXPY Test ===")
    # Run 3 discrete time steps
    for step in range(1, 4):
        print(f"--- Timestep {step} ---")

        # 1. Evolve Physics State (Manually calling update since manager wrapper is WIP)
        for i in range(manager._cbf_count):
            manager._cbf_buffer[i].update_state()

        # 2. Extract Linearizations
        lin_map = manager.get_cbf_linearizations(p_agent, p_next)

        cvxpy_constraints = []
        for name, expression in lin_map:
            # Prove the dynamic obstacle is actually moving in memory
            # (By manually peeking into the manager's buffer for the print statement)
            for i in range(manager._cbf_count):
                if manager._cbf_buffer[i].get_name() == name:
                    current_pos = manager._cbf_buffer[i].get_geometry().get_center().flatten()
                    print(f"[{name}] Pos: {current_pos} | Linearization Successful.")
            
            # Formulate the CP constraint: G * p_next + offset >= 0
            cvxpy_constraints.append(expression >= 0)

        # 3. Dummy CVXPY Problem to verify type compilation (Minimize movement distance)
        objective = cp.Minimize(cp.sum_squares(p_next - np.array(p_agent).flatten()))
        prob = cp.Problem(objective, cvxpy_constraints)

        try:
            prob.solve(solver=cp.OSQP) # OSQP is standard for quadratic MPC
            print(f"Solver Status: {prob.status} | p_next Output: {np.round(p_next.value, 3)}\n")
        except Exception as e:
            print(f"CRITICAL SOLVER ERROR: {e}\n")


    print("=== Phase 3: Swap and Pop Memory Verification ===")
    # Remove the static obstacle (Index 0). The dynamic obstacle (Index 1) should instantly drop into Index 0.
    manager.remove_cbf("Static_obstacle_1")
    print(f"Removed 'Static_obstacle_1'. Active CBFs Tracked: {manager._cbf_count}")

    swapped_name = manager.get_cbf_names()[0]
    print(f"Object at Buffer Index 0 is now: '{swapped_name}'")
    
    # Verify the trailing memory slot was wiped clean
    clean_slot_name = manager._cbf_buffer[1].get_name()
    print(f"Object at Buffer Index 1 (Trailing Slot) name is: '{clean_slot_name}' (Should be 'None')")
    print("\n=== Test Complete: Architecture is Stable! ===")

# Execute the test
run_cbf_architecture_test()

### CBF Deflection Mathematics Test

In [ ]:
def run_cbf_constraint_test():
    print("=== Initializing CBF Constraint Test ===\n")
    manager = CBFManager()

    # 1. Define Agent Start and Target
    # We add a tiny 0.1 Y-offset so the gradient pushes the agent AROUND the obstacle instead of deadlocking
    p_agent = np.array([[0.0], [0.1]]) 
    p_target = np.array([[5.0], [0.1]]) 
    step_size = 0.5 # How far the agent WANTS to move per timestep

    print("=== Phase 1: Injecting Barriers ===")
    
    # A. Static obstacle - Placed directly in the agent's path!
    geom_static = EllipticalGeometry(center=jnp.array([[2.0], [0.0]]), semi_major_axis=1.0)
    manager.add_cbf(
        name="Static_obstacle_1",
        geometry=geom_static,
        heartbeat=10.0,       
        physics_clock=10.0,   
        alpha=0.5,
        velocity=jnp.array([[0.0], [0.0]]) 
    )
    print(f"Added: Static_obstacle_1 at {geom_static.get_center().flatten()} (Radius: 1.0)")

    # B. Dynamic Obstacle (Moved out of the way just to test memory swap later)
    geom_dynamic = EllipticalGeometry(center=jnp.array([[10.0], [10.0]]), semi_major_axis=1.0)
    manager.add_cbf(
        name="Dynamic_Obstacle_1",
        geometry=geom_dynamic,
        heartbeat=10.0,
        physics_clock=10.0,
        alpha=0.8,
        velocity=jnp.array([[0.0], [1.0]]) 
    )
    print(f"Active CBFs Tracked: {manager._cbf_count}\n")


    print("=== Phase 2: Simulation Loop (Watch the Deflection!) ===")
    
    # Run 6 discrete time steps
    for step in range(1, 7):
        print(f"\n--- Timestep {step} ---")

        # 1. Evolve Physics State
        for i in range(manager._cbf_count):
            manager._cbf_buffer[i].update_state()

        # 2. Calculate the NOMINAL move (What the agent WANTS to do ignoring safety)
        direction = (p_target - p_agent)
        norm = np.linalg.norm(direction)
        if norm > 0:
            direction = direction / norm
        p_nominal = p_agent + direction * step_size

        # 3. Extract Linearizations & Build Constraints
        p_next = cp.Variable(2) 
        lin_map = manager.get_cbf_linearizations(jnp.array(p_agent), p_next)

        cvxpy_constraints = []
        for name, expression in lin_map:
            cvxpy_constraints.append(expression >= 0)

        # 4. CVXPY Problem: Try to reach p_nominal safely
        objective = cp.Minimize(cp.sum_squares(p_next - p_nominal.flatten()))
        prob = cp.Problem(objective, cvxpy_constraints)

        try:
            prob.solve(solver=cp.OSQP)
            if prob.status == 'optimal':
                # Convert solver output back to column vector for the next loop
                p_agent_next = p_next.value.reshape(2, 1) 
                
                print(f"Agent Pos:       {np.round(p_agent.flatten(), 3)}")
                print(f"Requested Move:  {np.round(p_nominal.flatten(), 3)}")
                print(f"CBF Constrained: {np.round(p_agent_next.flatten(), 3)}")
                
                # UPDATE the agent's position for the next frame
                p_agent = p_agent_next 
            else:
                print(f"Solver Status: {prob.status}")
        except Exception as e:
            print(f"CRITICAL SOLVER ERROR: {e}\n")


    print("\n=== Phase 3: Swap and Pop Memory Verification ===")
    manager.remove_cbf("Static_obstacle_1")
    print(f"Removed 'Static_obstacle_1'. Active CBFs Tracked: {manager._cbf_count}")
    print(f"Object at Buffer Index 0 is now: '{manager.get_cbf_names()[0]}'")
    print("=== Test Complete! ===")

# Execute the test
run_cbf_constraint_test()

## .gif Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import jax.numpy as jnp

def generate_smooth_cbf_gif():
    # 1. Setup Manager
    manager = CBFManager()
    
    # Add 2 Static Barriers (Red)
    manager.add_cbf("Static_0", EllipticalGeometry(jnp.array([[3.0], [5.0]]), 0.5), 10, 10, 0.5, jnp.array([[0.0], [0.0]]))
    manager.add_cbf("Static_1", EllipticalGeometry(jnp.array([[7.0], [3.0]]), 0.5), 10, 10, 0.5, jnp.array([[0.0], [0.0]]))
        
    # Add 3 Dynamic Barriers (Blue) 
    # Movement: Right and Diagonal-Lower-Right
    # vels: [x_vel, y_vel]
    dynamic_configs = [
        (jnp.array([[1.0], [8.0]]), jnp.array([[0.2], [0.0]])),   # Moving Right
        (jnp.array([[1.0], [5.0]]), jnp.array([[0.2], [-0.15]])), # Diagonal Lower-Right
        (jnp.array([[1.0], [2.0]]), jnp.array([[0.25], [-0.1]]))  # Diagonal Lower-Right
    ]
    
    for i, (pos, vel) in enumerate(dynamic_configs):
        manager.add_cbf(f"Dynamic_{i}", EllipticalGeometry(pos, 0.5), 10, 10, 0.8, vel)

    # 2. Setup Plot
    fig, ax = plt.subplots(figsize=(8, 8))
    
    def animate(frame):
        ax.clear()
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 10)
        ax.set_xlabel("X Position (m)")
        ax.set_ylabel("Y Position (m)")
        ax.set_title("Control Barrier Function Engine Test")
        ax.grid(True, linestyle='--', alpha=0.6)
        
        # Update physics
        for i in range(manager._cbf_count):
            manager._cbf_buffer[i].update_state()
            
            # Draw barrier
            center = manager._cbf_buffer[i].get_geometry().get_center().flatten()
            is_static = "Static" in manager._cbf_buffer[i].get_name()
            color = '#E74C3C' if is_static else '#3498DB' # Modern Red/Blue
            
            circle = plt.Circle((center[0], center[1]), 0.4, color=color, alpha=0.8)
            ax.add_patch(circle)
            
            # Annotate static barriers
            if is_static:
                ax.text(center[0], center[1]+0.3, 'Static', ha='center', fontsize=8, color='#C0392B')

    # 3. Generate Smooth GIF
    # interval=50ms (20fps) makes the motion look professional and smooth
    ani = animation.FuncAnimation(fig, animate, frames=100, interval=50, blit=False)
    ani.save('cbf_engine_test.gif', writer='pillow', fps=20)
    print("Smooth GIF saved as 'cbf_engine_test.gif'!")

generate_smooth_gif = generate_smooth_cbf_gif
generate_smooth_gif()